# Swin Object Detection — DIMER scaffold smoke notebook

**Profile:** `SMOKE`  
**Notebook spec:** DIMER Notebook Specification 1.0  
**Repository:** `kurtvalcorza/swin-detection-pipeline`  
**Lifecycle:** `scaffold`

This notebook verifies the repository's current **specification, lifecycle, model-card, and open-weight provenance scaffold**. The repository does not yet expose a runnable DIMER object-detection validator/finetuner pipeline, so this notebook intentionally does **not** claim `E2E` or `TASK-INFERENCE` conformance.

**By the end of this notebook you will be able to:** clone the exact reviewed scaffold revision; run the repository-owned fail-closed verifier; inspect the declared blockers and DIMER-hosting gate; and verify the pinned upstream checkpoint identity without downloading or deserializing it.

**This notebook does not demonstrate:** object-detection inference, fine-tuning, COCO AP evaluation, artifact production, serving, benchmark reproduction, or production fitness. Instance-mask output is outside the intended v1 object-detection contract.

## Prerequisites

- **Runtime:** Python 3.11+; CPU is sufficient.
- **Network:** GitHub access to clone the public repository revision pinned below.
- **Dependencies:** the executable path uses only the Python standard library plus Git; no model framework is installed because no task runtime exists.
- **Checkpoint trust boundary:** the canonical upstream checkpoint is a PyTorch `.pth` file. This smoke notebook never downloads or deserializes it; path/provenance consistency is not sender authenticity and does not make code-capable serialization safe to load.

## 1. Bootstrap the exact reviewed scaffold revision

The immutable revision below is the stacked lifecycle + model-card state under review. A successful checkout proves only source identity; later cells run the repository's own verifier against that source.

In [ ]:
from __future__ import annotations

import json
import os
import platform
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/kurtvalcorza/swin-detection-pipeline.git"
REPOSITORY_REF = "b384acf85b150f59b40aa833966aea3980f5c66e"
WORKSPACE = Path(os.environ.get("DIMER_TUTORIAL_WORKSPACE", "/content/swin-detection-pipeline"))
REPO_DIR = WORKSPACE / "repo"

def run(command, *, cwd=None):
    command = [str(x) for x in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)

WORKSPACE.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    run(["git", "clone", "--filter=blob:none", REPOSITORY, REPO_DIR])
run(["git", "fetch", "--quiet", "origin", REPOSITORY_REF], cwd=REPO_DIR)
run(["git", "checkout", "--detach", REPOSITORY_REF], cwd=REPO_DIR)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip()
if head != REPOSITORY_REF:
    raise RuntimeError(f"checked out {head}, expected {REPOSITORY_REF}")
print(json.dumps({"python": platform.python_version(), "repository": "kurtvalcorza/swin-detection-pipeline", "revision": head}, indent=2))

## 2. Run the repository-owned scaffold verifier

This is the repository's actual executable surface today. `PASS` means the declared scaffold, blockers, lifecycle, and provenance agree with the verifier's fail-closed rules. It does **not** mean an object detector executed.

In [ ]:
run([sys.executable, "scripts/verify_scaffold.py"], cwd=REPO_DIR)
print("Scaffold verification: PASS")

## 3. Inspect lifecycle, blockers, model-card presence, and hosting policy

The current scaffold must remain `lifecycle_status: scaffold`, emit no composition/release, and keep DIMER weight hosting blocked while redistribution status is unresolved. The model card is part of this reviewed scaffold state.

In [ ]:
surface = json.loads((REPO_DIR / "spec/pipeline-surface.json").read_text())
provenance = json.loads((REPO_DIR / "provenance/open-weights.json").read_text())
licensing = provenance["canonicalV1"]["weightLicensing"]
assert surface["dimerPipelineSpec"]["pipeline_spec"] == "1.0"
assert surface["dimerPipelineSpec"]["lifecycle_status"] == "scaffold"
assert surface["composition"]["manifestStatus"] == "NOT_EMITTED"
assert surface["composition"]["releaseStatus"] == "NOT_EMITTED"
assert surface["blockers"], "a scaffold with no runtime must not present an empty release gate"
assert licensing["redistribution_status"] == "unknown"
assert licensing["dimer_hosting"] == "BLOCKED"
assert (REPO_DIR / "MODEL_CARD.md").is_file()
print(json.dumps({"status": surface["status"], "lifecycle": surface["dimerPipelineSpec"]["lifecycle_status"], "blockers": surface["blockers"], "modelCardPresent": True, "weightRedistribution": licensing["redistribution_status"], "dimerHosting": licensing["dimer_hosting"]}, indent=2))

## 4. Assert the pinned checkpoint identity without loading it

The source of record is Swin-T + Mask R-CNN on COCO 2017. These assertions check the repository's pinned filename, SHA-256, and official task-repository revision. They are provenance checks, not runtime model evidence.

In [ ]:
canonical = provenance["canonicalV1"]
source = provenance["architectureSourceOfRecord"]
assert canonical["checkpointFile"] == "mask_rcnn_swin_tiny_patch4_window7_1x.pth"
assert canonical["checkpointSha256"] == "b67f9d6cd62a4d723c78faec1b49cbf548faa22437264defb00f2f6e54d21b78"
assert source["officialTaskRepository"] == "SwinTransformer/Swin-Transformer-Object-Detection"
assert source["referenceImplementationRevision"] == "7810b893902326d88068037477c848b551e2bd4e"
print(json.dumps({"checkpointFile": canonical["checkpointFile"], "checkpointSha256": canonical["checkpointSha256"], "digestStatus": canonical["checkpointDigestStatus"], "referenceImplementationRevision": source["referenceImplementationRevision"]}, indent=2))

## Interpretation and next step

A successful top-to-bottom run proves that this exact scaffold revision can be cloned; its repository-owned verifier passes; its lifecycle remains intentionally blocked; its scaffold model card is present; and the committed checkpoint provenance matches the expected pinned identity. It does **not** prove that object detection, training, evaluation, artifact export, or serving works.

This notebook remains **engineering-only**. Replace or upgrade it to a release-grade `E2E` or `TASK-INFERENCE` tutorial only after the canonical COCO representation, validator/finetuner releases, and accelerator-qualified task runtime exist. The release-grade notebook must then exercise that real API, validate COCO-style image/box inputs, report task-appropriate COCO metrics, run new-data inference where supported, export machine-readable detections/provenance, and carry clean-runtime execution evidence.